In [1]:
import csv

with open("lds-scriptures-2020.12.08/csv/lds-scriptures.csv", "r") as file:
    lines = csv.reader(file)

    documents = []
    metadata = []
    ids = []
    id = 1

    for i, line in enumerate(lines):
        if i == 0:
            continue

        documents.append(line[16])
        metadata.append({"verse_title": line[17]})
        ids.append(str(id))
        id += 1

In [ ]:
import openai
import chromadb
from chromadb.config import Settings
from tqdm import tqdm  # progress bar (pip install tqdm)
from httpx import RemoteProtocolError, ReadTimeout
import time

# --------- CONFIG ---------
OPENAI_MODEL = "text-embedding-3-small"   # Fastest high-quality embedding model
BATCH_SIZE = 300                        # Best speed: test 500 / 1000 / 2000 depending on your plan
# openai.api_key = "YOUR_OPENAI_API_KEY"   # Ensure this is set or use env var
RETRY_LIMIT = 5   # Max retries per batch


def get_chroma_client():
    return chromadb.CloudClient(
        api_key='ck-2ttjcu2DwFmXQHgtwU2f587uoUd2DT2sxJv4rLMEsQdY',
        tenant='cd2d8da5-72c1-4ece-8555-c0fb9d1e4e2a',
        database='test'
    )

In [3]:
# --------- EMBEDDING FUNCTION (BATCH) ---------
def embed_batch(text_list):
    response = openai.embeddings.create(
        model=OPENAI_MODEL,
        input=text_list  # Can handle large batch (OpenAI recommends <= 2048 tokens per input item)
    )
    return [item.embedding for item in response.data]

In [8]:
def embed_query(query):
    response = openai.embeddings.create(
        model=OPENAI_MODEL,
        input=query,
    )
    return response.data[0].embedding
        

In [ ]:
client = get_chroma_client()
collection = client.get_or_create_collection(name="scriptures")

total = len(documents)
batches = range(0, total, BATCH_SIZE)

for batch_index, i in enumerate(tqdm(batches, desc="Ingesting into Chroma (Resume-Safe)")):
    batch_docs = documents[i:i + BATCH_SIZE]
    batch_ids = ids[i:i + BATCH_SIZE]
    batch_meta = metadata[i:i + BATCH_SIZE]

    # Retry loop
    for attempt in range(RETRY_LIMIT):
        try:
            # Step 1: Pre-embed
            batch_embs = embed_batch(batch_docs)

            # Step 2: Insert into Chroma
            collection.add(
                ids=batch_ids,
                documents=batch_docs,
                metadatas=batch_meta,
                embeddings=batch_embs
            )

            print(f"✅ Successfully inserted batch {batch_index + 1}/{len(batches)}")
            break  # Exit retry loop on success

        except (RemoteProtocolError, ReadTimeout, Exception) as e:
            wait_time = 2 ** attempt  # Exponential backoff: 1s, 2s, 4s, 8s...
            print(f"⚠️ Error on batch {batch_index + 1}: {e}")
            print(f"⏳ Retrying in {wait_time} seconds... (Attempt {attempt + 1}/{RETRY_LIMIT})")
            time.sleep(wait_time)

            # Optional: reconnect client every few retries to reset connections
            if attempt == 2:  # Reconnect midway if failing repeatedly
                print("🔄 Reconnecting CloudClient to avoid stale connection...")
                client = get_chroma_client()
                collection = client.get_or_create_collection(name="scriptures")

    else:
        print(f"❌ Failed to insert batch {batch_index + 1} after {RETRY_LIMIT} retries. Skipping and continuing...")
        continue

print("🎉 Ingestion completed with retry & resume protection!")

Ingesting into Chroma (Resume-Safe):   1%|          | 1/140 [00:03<07:24,  3.20s/it]

✅ Successfully inserted batch 1/140


Ingesting into Chroma (Resume-Safe):   1%|▏         | 2/140 [00:04<04:40,  2.04s/it]

✅ Successfully inserted batch 2/140
⚠️ Error on batch 3: Server disconnected without sending a response.
⏳ Retrying in 1 seconds... (Attempt 1/5)


Ingesting into Chroma (Resume-Safe):   2%|▏         | 3/140 [00:08<07:09,  3.13s/it]

...

✅ Successfully inserted batch 138/140


Ingesting into Chroma (Resume-Safe):  99%|█████████▉| 139/140 [05:06<00:02,  2.20s/it]

✅ Successfully inserted batch 139/140


Ingesting into Chroma (Resume-Safe): 100%|██████████| 140/140 [05:07<00:00,  2.20s/it]

✅ Successfully inserted batch 140/140
🎉 Ingestion completed with retry & resume protection!


In [18]:
userInput = input("Enter something I'll return a few scriptures about it.")
n = 3
resp =collection.query(
    query_embeddings=[embed_query(userInput)],
    n_results=n,
    include=["documents", "metadatas"]
)

for script in range(n):
    print(resp["metadatas"][0][script]['verse_title'])
    print(resp["documents"][0][script])
    print("\n")
    

resp


Zechariah 12:14
All the families that remain, every family apart, and their wives apart.


3 Nephi 18:21
Pray in your families unto the Father, always in my name, that your wives and your children may be blessed.


2 Nephi 2:20
And they have brought forth children; yea, even the family of all the earth.




{'ids': [['23060', '36545', '31772']],
 'distances': None,
 'embeddings': None,
 'metadatas': [[{'verse_title': 'Zechariah 12:14'},
   {'verse_title': '3 Nephi 18:21'},
   {'verse_title': '2 Nephi 2:20'}]],
 'documents': [['All the families that remain, every family apart, and their wives apart.',
   'Pray in your families unto the Father, always in my name, that your wives and your children may be blessed.',
   'And they have brought forth children; yea, even the family of all the earth.']],
 'uris': None,
 'data': None,
 'included': ['documents', 'metadatas']}